In [1]:
import io
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

csv_path = Path("data") / "freMTPL2freq.csv"
if not csv_path.exists():
    url = "https://github.com/ds-careers-ominimo/ominimo-careers-data/raw/main/ml_claims_forecasting.zip"
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    csv_path.write_bytes(archive.read("freMTPL2freq.csv"))

claims = pd.read_csv(csv_path, dtype={"IDpol": "Int64"})
print(claims.shape)

(678013, 12)


In [2]:
feature_columns = ["Area", "VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Density", "Region"]

print(claims.isna().sum())
print(f"IDpol duplicated {claims['IDpol'].duplicated().sum()} unique {claims['IDpol'].nunique()} rows {len(claims)}")
print(claims["ClaimNb"].value_counts().sort_index())
print(f"ClaimNb zero share {(claims['ClaimNb'] == 0).mean():.4f} max {claims['ClaimNb'].max()} above 4 {(claims['ClaimNb'] > 4).sum()}")
print(claims["Exposure"].agg(["min", "median", "mean", "max"]))
print(f"Exposure above 1 {(claims['Exposure'] > 1).sum()} below 0.02 {(claims['Exposure'] < 0.02).sum()} equal to 2 {(claims['Exposure'] == 2).sum()}")
duplicated_features = claims.duplicated(subset=feature_columns, keep=False).sum()
duplicated_with_exposure = claims.duplicated(subset=feature_columns + ["Exposure"], keep=False).sum()
print(f"duplicate rows on 9 feature columns {duplicated_features} share {duplicated_features / len(claims):.3f}")
print(f"duplicate rows on 9 feature columns plus Exposure {duplicated_with_exposure} share {duplicated_with_exposure / len(claims):.3f}")
print(claims[["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]].agg(["min", "max"]))
print(f"VehAge above 30 {(claims['VehAge'] > 30).sum()} DrivAge above 90 {(claims['DrivAge'] > 90).sum()} BonusMalus above 150 {(claims['BonusMalus'] > 150).sum()}")
implied_frequency = claims["ClaimNb"] / claims["Exposure"]
print(f"implied frequency max {implied_frequency.max():.1f} above 20 {(implied_frequency > 20).sum()}")

exposure_band = pd.cut(claims["Exposure"], bins=[0, 0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 1.0, 2.1])
frequency_by_exposure = (
    claims.assign(ExposureBand=exposure_band)
    .groupby("ExposureBand", observed=False)
    .agg(row_count=("IDpol", "size"), claim_count=("ClaimNb", "sum"), exposure_sum=("Exposure", "sum"))
)
frequency_by_exposure["frequency"] = frequency_by_exposure["claim_count"] / frequency_by_exposure["exposure_sum"]
print(frequency_by_exposure)

portfolio_frequency = claims["ClaimNb"].sum() / claims["Exposure"].sum()
print(f"mean(ClaimNb / Exposure) {implied_frequency.mean():.6f} vs sum(ClaimNb) / sum(Exposure) {portfolio_frequency:.6f}")

density_area_counts = claims.groupby(["Density", "Area"]).size()
density_levels = claims["Density"].nunique()
density_levels_with_multiple_areas = density_area_counts.groupby(level="Density").size().gt(1).sum()
density_majority_accuracy = density_area_counts.groupby(level="Density").max().sum() / len(claims)
print(f"Density levels {density_levels}, levels mapping to multiple Area values {density_levels_with_multiple_areas}, majority map accuracy {density_majority_accuracy:.6f}")

IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64
IDpol duplicated 0 unique 678013 rows 678013
ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
5          2
6          1
8          1
9          1
11         3
16         1
Name: count, dtype: int64


ClaimNb zero share 0.9498 max 16 above 4 9


min       0.002732
median    0.490000
mean      0.528750
max       2.010000
Name: Exposure, dtype: float64
Exposure above 1 1224 below 0.02 13603 equal to 2 1


duplicate rows on 9 feature columns 257911 share 0.380
duplicate rows on 9 feature columns plus Exposure 41505 share 0.061
     VehPower  VehAge  DrivAge  BonusMalus  Density
min         4       0       18          50        1
max        15     100      100         230    27000
VehAge above 30 1116 DrivAge above 90 401 BonusMalus above 150 209
implied frequency max 732.0 above 20 1214


              row_count  claim_count   exposure_sum  frequency
ExposureBand                                                  
(0.0, 0.01]       13603          362     101.245463   3.575469
(0.01, 0.02]       5656          173     113.120000   1.529349
(0.02, 0.05]      30083          920    1152.920000   0.797974
(0.05, 0.1]       78254         2005    6162.850000   0.325336
(0.1, 0.25]       95216         3671   17434.630000   0.210558
(0.25, 0.5]      131302         6481   51388.740000   0.126117
(0.5, 1.0]       322675        22436  280782.600000   0.079905
(1.0, 2.1]         1224           54    1363.340000   0.039609
mean(ClaimNb / Exposure) 0.263964 vs sum(ClaimNb) / sum(Exposure) 0.100703


Density levels 1607, levels mapping to multiple Area values 3, majority map accuracy 0.998376


Exposure assumption

Although the brief states the exposure period is one or two years, the data show a continuous duration. The median is 0.49 years, the range is from 0.0027 to 2.01, and only one entry is exactly 2. I thus regard exposure as the time at risk in years and calculate the annual claim frequency as ClaimNb divided by Exposure.

The annualised frequency drops from 3.575 for exposure up to 0.01 to 0.040 when exposure exceeds 1. This casts doubt on the assumption of a duration-neutral rate, but the marginal pattern alone cannot show whether the effect is due to duration or a changing risk mix. One possible reason might be claim-related early termination. However, this dataset does not include policy dates or termination reasons, so it cannot be tested.

In [3]:
import numpy as np

expected_claims = claims["Exposure"] * portfolio_frequency
baseline_deviance = 2 * expected_claims.copy()
positive_claims = claims["ClaimNb"] > 0
baseline_deviance.loc[positive_claims] = 2 * (
    claims.loc[positive_claims, "ClaimNb"]
    * np.log(claims.loc[positive_claims, "ClaimNb"] / expected_claims.loc[positive_claims])
    - claims.loc[positive_claims, "ClaimNb"]
    + expected_claims.loc[positive_claims]
)
total_deviance = baseline_deviance.sum()
top_20 = baseline_deviance.nlargest(20).index
cleaning_evidence = pd.DataFrame(
    [
        ["ClaimNb > 4", (claims["ClaimNb"] > 4).sum(), baseline_deviance[claims["ClaimNb"] > 4].sum() / total_deviance],
        ["Exposure < 0.02", (claims["Exposure"] < 0.02).sum(), baseline_deviance[claims["Exposure"] < 0.02].sum() / total_deviance],
        ["Exposure > 1", (claims["Exposure"] > 1).sum(), baseline_deviance[claims["Exposure"] > 1].sum() / total_deviance],
        ["Top 20 by deviance", len(top_20), baseline_deviance.loc[top_20].sum() / total_deviance],
    ],
    columns=["group", "row_count", "deviance_share"],
).set_index("group")
print(f"Full-portfolio intercept-only Poisson deviance {total_deviance:.3f}")
print(cleaning_evidence)
print(f"Density median {claims['Density'].median():.0f} max {claims['Density'].max()}")

Full-portfolio intercept-only Poisson deviance 224481.150
                    row_count  deviance_share
group                                        
ClaimNb > 4                 9        0.003717
Exposure < 0.02         13603        0.020312
Exposure > 1             1224        0.001825
Top 20 by deviance         20        0.005391
Density median 393 max 27000


Cleaning decisions

The prepared dataset keeps all the original rows and values that were provided.

* Nothing is imputed and no rows are removed: there are neither missing values nor duplicate IDpol values. Although feature profiles repeat, they can refer to different policies.
* There is no limit on ClaimNb. The nine rows with more than four claims make up 0.372% of the full-portfolio baseline deviance, and setting a cap would change the target.
* Rows with `Exposure` below 0.02 are kept; they make up 2.031% of the baseline deviance, and their unusual annualised frequency will be examined later.
* The 1,224 rows in which Exposure is above one are kept since the available fields do not indicate that these durations are in error.
* `Density` is unchanged in the prepared data; its later log transformation is model preprocessing for a skewed distribution, not target cleaning. `Area` and `Density` are both retained, but their near-redundancy should be considered when interpreting the models.



In [4]:
prepared_path = Path("data") / "claims_prepared.parquet"
claims.to_parquet(prepared_path)
prepared = pd.read_parquet(prepared_path)
print(prepared_path)
print(prepared.shape)
print(prepared["IDpol"].dtype)

data\claims_prepared.parquet
(678013, 12)
Int64
